3) 대화 토큰 버퍼 메모리

In [ ]:
# [목적] ConversationTokenBufferMemory로 토큰 수를 기준으로 최근 대화를 관리하는 예제
# 모델과 최대 토큰 수를 설정한 메모리를 만들고, 이후 문답이 한도를 넘으면 오래된 내용을 제외합니다.
# 모델 입력 길이를 제한하면서도 가능한 최근 문맥을 유지할 때 사용하는 방식입니다.
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

# ChatOpenAI는 OpenAI 채팅 모델을 LangChain에서 사용할 수 있게 만드는 객체입니다.
llm = ChatOpenAI(model_name="gpt-4o")  # LLM 모델 생성

# max_token_limit은 메모리에 남길 대화의 최대 토큰 수이며, 토큰은 모델이 텍스트를 처리하는 단위입니다.
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=150,
    return_messages=True,  # 최대 토큰 길이 제한
)

In [ ]:
# [목적] 여러 설치 안내 문답을 토큰 제한 메모리에 순서대로 저장하는 예제
# save_context를 반복 호출하면 새 문답이 추가되고, 총 토큰 수가 한도를 넘을 때 가장 오래된 기록부터 제거됩니다.
# 긴 대화에서도 최신 안내 내용을 모델에 전달할 수 있도록 메모리를 채웁니다.
memory.save_context(
    inputs={"human": "안녕하세요, 저는 ... 설치 방법을 알려주실 수 있나요?"},
    outputs={"ai": "안녕하세요! ... 해당 기계 모델 번호를 알려주시겠어요?"},
)

memory.save_context(
    inputs={"human": "네, 모델 번호는 XG-200입니다."},
    outputs={"ai": "감사합니다. XG-200 모델의 설치 안내를 ..."},
)

memory.save_context(
    inputs={"human": "전원은 확인했습니다. 다음 단계는 무엇인가요?"},
    outputs={"ai": "좋습니다. 다음으로, 기계를 평평하고 안정된 바닥에 배치해 ..."},
)

memory.save_context(
    inputs={"human": "연결은 어떻게 하나요?"},
    outputs={"ai": "매뉴얼의 5페이지를 참조해 주세요. 케이블 연결에 관한 ..."},
)

memory.save_context(
    inputs={"human": "설치가 완료되면 어떻게 해야 하나요?"},
    outputs={"ai": "설치가 완료되면, 전원을 켜고 초기 구동 테스트를 ..."},
)

memory.save_context(
    inputs={"human": "감사합니다, 도움이 많이 되었어요!"},
    outputs={"ai": "언제든지 도와드릴 준비가 되어 있습니다. 추가적인 ..."},
)

In [ ]:
# [목적] 토큰 한도 안에 남아 있는 최신 대화 이력을 확인하는 예제
# history를 조회하면 오래된 문답은 제외되고 최대 150토큰 범위에서 유지된 메시지 목록이 반환됩니다.
# 토큰 기준으로 메모리가 실제로 정리되었는지 확인하는 데 사용합니다.
memory.load_memory_variables({})["history"]